In [1]:
"""
End-to-end prototype with **on-the-fly noise injection** (noiser is applied directly in the loop):
1) Train segmentation on NOISY(images) vs CLEAN-derived masks.
2) Detect peaks (multiple images) via simple NMS on segmentation logits.
3) Tokenize each detected image (variable-length sequence per system) with multi-scale crops + geometric features.
4) Train a lightweight Transformer classifier for substructure (0/1).

Assumptions about loaders (adapt if needed):
- CLEAN train loader yields either (clean_imgs,) or (clean_imgs, sub_labels). For segmentation you can use either; for classifier you need labels.
- CLEAN val loader yields (clean_imgs, sub_labels).
- Images are [B,1,80,80] floats ~[0,1].
- You have a callable `noiser` so that `noisy = noiser(clean)` returns a batch-shaped tensor on the SAME device.

Dependencies: torch, torchvision (no opencv/skimage required)
"""

import math
from dataclasses import dataclass
from typing import List, Tuple
import os
import gc
import time
import json
import argparse
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from typing import Callable, List

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import wandb

from deep_learning.NN_datasets import NoNoiseDataset
from deep_learning.NN_datasets.dataloaders import custom_dataloader
from deep_learning.NN_models import ResNet50
from noise_applicator.noisers.base_noiser import BaseNoiser, EuclidNoiser
from config import TRAINED_CLASSIFIERS_DIR
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

# -------------------------------
# 1) Tiny U-Net-ish segmentation model
# -------------------------------

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class SmallUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base)
        self.enc2 = DoubleConv(base, base*2)
        self.enc3 = DoubleConv(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.dec2 = DoubleConv(base*4 + base*2, base*2)
        self.dec1 = DoubleConv(base*2 + base, base)
        self.out = nn.Conv2d(base, out_ch, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool(e1)
        e2 = self.enc2(p1)
        p2 = self.pool(e2)
        e3 = self.enc3(p2)
        up2 = F.interpolate(e3, scale_factor=2, mode='bilinear', align_corners=False)
        d2 = self.dec2(torch.cat([up2, e2], dim=1))
        up1 = F.interpolate(d2, scale_factor=2, mode='bilinear', align_corners=False)
        d1 = self.dec1(torch.cat([up1, e1], dim=1))
        logits = self.out(d1)
        return logits  # [B,1,H,W]

# -------------------------------
# 2) CLEAN→mask utility
# -------------------------------

def clean_to_mask(clean_imgs: torch.Tensor, thresh: float = 0.2) -> torch.Tensor:
    """Simple thresholding + tiny closing to get masks from CLEAN images."""
    with torch.no_grad():
        masks = (clean_imgs > thresh).float()
        masks = F.max_pool2d(masks, kernel_size=3, stride=1, padding=1)
    return masks

# -------------------------------
# 3) On-the-fly noise iterator & segmentation training
# -------------------------------

def iter_clean_with_noiser(clean_loader: DataLoader, noiser, device: str):
    """Yield (noisy_imgs, sub_labels, clean_imgs) created from CLEAN loader on the fly.
    Supports batch shapes (clean,) or (clean, sub_labels).
    """
    for batch in clean_loader:
        if isinstance(batch, (list, tuple)):
            if len(batch) == 2:
                clean_imgs, sub_labels = batch
            elif len(batch) == 1:
                (clean_imgs,) = batch
                sub_labels = None
            else:
                raise ValueError("Unsupported batch structure from clean_loader")
        else:
            clean_imgs, sub_labels = batch, None
        clean_imgs = clean_imgs.to(device)
        with torch.no_grad():
            noisy_imgs = noiser(clean_imgs)
        yield noisy_imgs, sub_labels, clean_imgs

@dataclass
class SegTrainCfg:
    epochs: int = 5
    lr: float = 1e-3
    threshold_clean: float = 0.2
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'


def train_segmentation(model: nn.Module,
                       clean_loader: DataLoader,
                       noiser,
                       cfg: SegTrainCfg = SegTrainCfg()):
    model = model.to(cfg.device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    bce = nn.BCEWithLogitsLoss()

    model.train()
    for epoch in range(cfg.epochs):
        running = 0.0
        n_pix = 0
        for noisy, _, clean in iter_clean_with_noiser(clean_loader, noiser, cfg.device):
            target = clean_to_mask(clean, cfg.threshold_clean)
            logits = model(noisy)
            loss = bce(logits, target)
            opt.zero_grad(); loss.backward(); opt.step()
            running += loss.item() * noisy.size(0)
            n_pix += noisy.size(0)
        print(f"Seg Epoch {epoch+1}/{cfg.epochs} - loss={running/max(1,n_pix):.4f}")
    return model

# -------------------------------
# 4) Peak detection via NMS on segmentation logits
# -------------------------------

@dataclass
class PeakCfg:
    prob_thresh: float = 0.3  # threshold on sigmoid(logits)
    nms_kernel: int = 5       # local window for NMS (odd)
    max_peaks: int = 6        # maximum images expected per system


def detect_peaks_from_logits(logits: torch.Tensor, cfg: PeakCfg) -> List[List[Tuple[int,int,float]]]:
    with torch.no_grad():
        probs = torch.sigmoid(logits)  # [B,1,H,W]
        B, _, H, W = probs.shape
        pad = cfg.nms_kernel // 2
        pooled = F.max_pool2d(probs, kernel_size=cfg.nms_kernel, stride=1, padding=pad)
        is_peak = (probs == pooled) & (probs > cfg.prob_thresh)
        peaks_per_img: List[List[Tuple[int,int,float]]] = []
        for b in range(B):
            ys, xs = torch.nonzero(is_peak[b,0], as_tuple=True)
            scores = probs[b,0,ys,xs]
            if scores.numel() > 0:
                vals, idx = torch.sort(scores, descending=True)
                idx = idx[:cfg.max_peaks]
                ys, xs, vals = ys[idx], xs[idx], vals[idx]
                peaks = [(int(ys[i].item()), int(xs[i].item()), float(vals[i].item())) for i in range(len(idx))]
            else:
                peaks = []
            peaks_per_img.append(peaks)
        return peaks_per_img

# -------------------------------
# 5) Tokenization: multi-scale patch encoder + geometric features
# -------------------------------

@dataclass
class TokenCfg:
    crop_sizes: Tuple[int, ...] = (9, 15)
    token_dim: int = 128
    max_tokens: int = 32

class BlobTokenizer(nn.Module):
    def __init__(self, cfg: TokenCfg):
        super().__init__()
        self.cfg = cfg
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.to_token = nn.Linear(64 + 6, cfg.token_dim)  # +6 geom features
        self.cls = nn.Parameter(torch.randn(1, 1, cfg.token_dim))
        self.sep = nn.Parameter(torch.randn(1, 1, cfg.token_dim))

    @staticmethod
    def _grid_crop(img: torch.Tensor, cy: float, cx: float, size: int) -> torch.Tensor:
        H, W = img.shape[-2:]
        ys = torch.linspace(-(size-1)/2, (size-1)/2, size, device=img.device)
        xs = torch.linspace(-(size-1)/2, (size-1)/2, size, device=img.device)
        grid_y, grid_x = torch.meshgrid(ys, xs, indexing='ij')
        grid_y = (grid_y + cy) / (H-1) * 2 - 1
        grid_x = (grid_x + cx) / (W-1) * 2 - 1
        grid = torch.stack([grid_x, grid_y], dim=-1).unsqueeze(0)
        patch = F.grid_sample(img.unsqueeze(0), grid, mode='bilinear', align_corners=True)
        return patch.squeeze(0)

    def forward(self, imgs: torch.Tensor, peaks_batch: List[List[Tuple[int,int,float]]]):
        B, _, H, W = imgs.shape
        device = imgs.device
        toks = []
        for b in range(B):
            peaks = peaks_batch[b]
            if len(peaks) > 0:
                ys = torch.tensor([p[0] for p in peaks], device=device, dtype=torch.float32)
                xs = torch.tensor([p[1] for p in peaks], device=device, dtype=torch.float32)
                cy, cx = ys.mean(), xs.mean()
                r = torch.sqrt(((ys - cy)**2 + (xs - cx)**2).mean() + 1e-6)
            else:
                cy, cx, r = torch.tensor(40., device=device), torch.tensor(40., device=device), torch.tensor(20., device=device)
            seq = [self.cls.expand(1, -1, -1)]
            for (yy, xx, sc) in peaks:
                patches = []
                for size in self.cfg.crop_sizes:
                    patch = self._grid_crop(imgs[b,0], float(yy), float(xx), size)
                    patches.append(patch)
                largest = max(self.cfg.crop_sizes)
                res = []
                for p in patches:
                    if p.shape[-1] != largest:
                        p = F.interpolate(p.unsqueeze(0), size=(largest, largest), mode='bilinear', align_corners=False).squeeze(0)
                    res.append(p)
                stack = torch.stack(res, dim=0).mean(0)
                feat = self.enc(stack.unsqueeze(0)).flatten(1)  # [1,64]
                dy = (yy - cy) / (r + 1e-6)
                dx = (xx - cx) / (r + 1e-6)
                rho = (dy*dy + dx*dx) ** 0.5
                ang = math.atan2(float(dy), float(dx))
                geom = torch.tensor([[dy, dx, rho, math.sin(ang), math.cos(ang), sc]], device=device, dtype=torch.float32)
                token_vec = self.to_token(torch.cat([feat, geom], dim=1)).unsqueeze(1)
                seq.append(token_vec)
                seq.append(self.sep.expand(1, -1, -1))
            seq_tensor = torch.cat(seq, dim=1)
            toks.append(seq_tensor)
        maxT = min(max(t.size(1) for t in toks), self.cfg.max_tokens)
        out = torch.zeros(len(toks), maxT, self.cfg.token_dim, device=device)
        attn = torch.zeros(len(toks), maxT, device=device)
        for b in range(len(toks)):
            t = toks[b]; T = min(t.size(1), maxT)
            out[b, :T] = t[:, :T, :]
            attn[b, :T] = 1
        return out, attn

# -------------------------------
# 6) Transformer classifier
# -------------------------------

class TransformerClassifier(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=3, dim_feedforward=256, dropout=0.1):
        super().__init__()
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                               dim_feedforward=dim_feedforward,
                                               dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.cls_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, tokens: torch.Tensor, attn_mask: torch.Tensor):
        key_padding_mask = (attn_mask == 0)
        x = self.encoder(tokens, src_key_padding_mask=key_padding_mask)
        cls = x[:, 0, :]
        return self.cls_head(cls).squeeze(-1)

# -------------------------------
# 7) Classifier training (seg frozen) with on-the-fly noiser
# -------------------------------

@dataclass
class ClsTrainCfg:
    epochs: int = 5
    lr: float = 1e-3
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    peak: PeakCfg = PeakCfg()
    tok: TokenCfg = TokenCfg()


def train_classifier(seg_model: nn.Module,
                     tokenizer: BlobTokenizer,
                     clf: TransformerClassifier,
                     clean_loader: DataLoader,
                     noiser,
                     cfg: ClsTrainCfg = ClsTrainCfg()):
    device = cfg.device
    seg_model = seg_model.to(device).eval()
    for p in seg_model.parameters():
        p.requires_grad = False
    tokenizer = tokenizer.to(device)
    clf = clf.to(device)

    opt = torch.optim.Adam(list(tokenizer.parameters()) + list(clf.parameters()), lr=cfg.lr)
    bce = nn.BCEWithLogitsLoss()

    for epoch in range(cfg.epochs):
        tokenizer.train(); clf.train()
        running = 0.0; n = 0
        for batch in clean_loader:
            if not (isinstance(batch, (list, tuple)) and len(batch) == 2):
                raise ValueError("clean_loader must yield (clean_imgs, sub_labels) for classifier training")
            clean, sub_labels = batch
            clean = clean.to(device)
            sub_labels = sub_labels.float().to(device)
            with torch.no_grad():
                noisy = noiser(clean)
                seg_logits = seg_model(noisy)
                peaks = detect_peaks_from_logits(seg_logits, cfg.peak)
            tokens, attn = tokenizer(noisy, peaks)
            pred = clf(tokens, attn)
            loss = bce(pred, sub_labels)
            opt.zero_grad(); loss.backward(); opt.step()
            running += loss.item() * noisy.size(0); n += noisy.size(0)
        print(f"Cls Epoch {epoch+1}/{cfg.epochs} - loss={running/max(1,n):.4f}")
    return tokenizer, clf

# -------------------------------
# 8) Evaluation helper
# -------------------------------

def evaluate(seg_model: nn.Module,
             tokenizer: BlobTokenizer,
             clf: TransformerClassifier,
             clean_loader: DataLoader,
             noiser,
             peak_cfg: PeakCfg = PeakCfg(),
             device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
    seg_model.eval(); tokenizer.eval(); clf.eval()
    sigm = nn.Sigmoid()
    ys = []; ps = []
    with torch.no_grad():
        for clean, labels in clean_loader:
            clean = clean.to(device)
            labels = labels.float().to(device)
            noisy = noiser(clean)
            logits = seg_model(noisy)
            peaks = detect_peaks_from_logits(logits, peak_cfg)
            tokens, attn = tokenizer(noisy, peaks)
            prob = sigm(clf(tokens, attn))
            ys.append(labels.cpu()); ps.append(prob.cpu())
    y = torch.cat(ys); p = torch.cat(ps)
    acc = ((p>0.5).float() == y).float().mean().item()
    print(f"Eval accuracy@0.5 = {acc:.3f}")
    return acc

# -------------------------------
# 9) Usage skeleton (pseudo-code)
# -------------------------------
# You provide your own CLEAN loaders and the noiser.
# Example shape expectation:
#   train_clean_loader: yields (clean_imgs, sub_labels)   # labels only needed for classifier stage
#   val_clean_loader:   yields (clean_imgs, sub_labels)
#   noiser: callable, noiser(clean_imgs) -> noisy_imgs (same device/shape)

#raise SystemExit("Plug your CLEAN DataLoaders + noiser, then call train_segmentation/train_classifier/evaluate.")





train_dataset = NoNoiseDataset(
            "min_mass_10e9",
            grid_pixel_side=80,
            grid_width_arcsec=8.0,
            broadcasting=True,
            samples_used=200_000,
            upscaling=5,
)
test_dataset = NoNoiseDataset(
            "min_mass_10e9_test",
            grid_pixel_side=80,
            grid_width_arcsec=8.0,
            broadcasting=True,
            samples_used=100_000,
            upscaling=5,
)


train_clean_loader = custom_dataloader(
    train_dataset, batch_size=256
)
val_clean_loader = custom_dataloader(
    test_dataset, batch_size=256
)

#Example:
noiser = EuclidNoiser(); noiser.set_device("cuda")
seg = SmallUNet(); seg = train_segmentation(seg, train_clean_loader, noiser)
tok = BlobTokenizer(TokenCfg(crop_sizes=(9,15), token_dim=128, max_tokens=32))
clf = TransformerClassifier(d_model=128, nhead=4, num_layers=3)
tok, clf = train_classifier(seg, tok, clf, train_clean_loader, noiser)
evaluate(seg, tok, clf, val_clean_loader, noiser)


cpu
Using broadcasting mode
Using device: cuda
Currently this dataloader is calculating the images in float32
Using broadcasting mode
Using device: cuda
Currently this dataloader is calculating the images in float32
200000
100000
Seg Epoch 1/5 - loss=0.0055
Seg Epoch 2/5 - loss=0.0000
Seg Epoch 3/5 - loss=0.0000
Seg Epoch 4/5 - loss=0.0000
Seg Epoch 5/5 - loss=0.0000
Cls Epoch 1/5 - loss=0.6961
Cls Epoch 2/5 - loss=0.6935
Cls Epoch 3/5 - loss=0.6934
Cls Epoch 4/5 - loss=0.6933
Cls Epoch 5/5 - loss=0.6933


/raven/u/fcitterio/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:505: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Eval accuracy@0.5 = 0.500


0.4996899962425232

In [ ]:
train_dataset = NoNoiseDataset(
            "min_mass_10e9",
            grid_pixel_side=80,
            grid_width_arcsec=8.0,
            broadcasting=True,
            samples_used=2_000_000,
            upscaling=5,
)

train_clean_loader = custom_dataloader(
    train_dataset, batch_size=1024
)

tok = BlobTokenizer(TokenCfg(crop_sizes=(9,15), token_dim=128, max_tokens=32))
clf = TransformerClassifier(d_model=128, nhead=4, num_layers=3)
tok, clf = train_classifier(seg, tok, clf, train_clean_loader, noiser)
evaluate(seg, tok, clf, val_clean_loader, noiser)

Using broadcasting mode
Using device: cuda
Currently this dataloader is calculating the images in float32
2000000
Cls Epoch 1/5 - loss=0.6940
